In [1]:
import numpy as np
import os

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import direction_utils as utils

In [2]:
class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=32):
        super(SEBlock, self).__init__()
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)  # Output size: (batch_size, in_channels, 1, 1)
        
        # self.fc1 = nn.Linear(in_channels, max(1, in_channels // reduction), bias=False)  # Squeeze
        self.fc1 = nn.Linear(in_channels, in_channels // reduction, bias=False)  # Squeeze

        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(in_channels // reduction, in_channels, bias=False)  # Excitation
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        batch_size, channels, height, width = x.size()
        
        # Squeeze: Global Average Pooling
        out = self.global_avg_pool(x).view(batch_size, channels)  # Shape: (batch_size, in_channels)
        
        # Excitation: Fully connected layers
        out = self.fc1(out)  # Shape: (batch_size, in_channels // reduction)
        out = self.relu(out)
        out = self.fc2(out)  # Shape: (batch_size, in_channels)
        out = self.sigmoid(out).view(batch_size, channels, 1, 1)  # Reshape to (batch_size, in_channels, 1, 1)
        
        # Scale the input by the SE weights
        return x * out.expand_as(x)

In [3]:
class EEGNet(nn.Module):
    def __init__(self, nb_classes, Chans=27, Samples=2500, dropoutRate=0.5, 
                 kernLength=64, F1=8, D=2, F2=16, norm_rate=0.25, dropoutType='Dropout'):
        super(EEGNet, self).__init__()
        
        # Handle dropout type
        if dropoutType == 'SpatialDropout2D':
            self.dropout = nn.Dropout2d(dropoutRate)
        elif dropoutType == 'Dropout':
            self.dropout = nn.Dropout(dropoutRate)
        else:
            raise ValueError('dropoutType must be one of SpatialDropout2D or Dropout.')

        # Block 1
        self.conv1 = nn.Conv2d(1, F1, (1, kernLength), padding='same', bias=False)
        self.batchnorm1 = nn.BatchNorm2d(F1)
        # Squeeze-and-Excitation Block
        self.se1 = SEBlock(F1, 32)


        self.depthwiseConv = nn.Conv2d(F1, F1*D, (Chans, 1), groups=F1, bias=False)
        self.batchnorm2 = nn.BatchNorm2d(F1*D)
        self.se2 = SEBlock(F1*D, 32)  # Squeeze-and-Excitation Block
        self.pool1 = nn.AvgPool2d((1, 4))

        # Block 2
        self.separableConv = nn.Conv2d(F1*D, F2, (1, 16), padding='same', bias=False)
        self.batchnorm3 = nn.BatchNorm2d(F2)
        self.se3 = SEBlock(F2, 32)  # Squeeze-and-Excitation Block
        self.pool2 = nn.AvgPool2d((1, 8))

        # Flatten and Dense
        self.flatten = nn.Flatten()
        self.dense = nn.Linear(F2 * (Samples // (4 * 8)), nb_classes)
        self.norm_constraint = nn.utils.weight_norm(self.dense)

    def forward(self, x):
        # Block 1
        x = self.conv1(x)
        x = self.batchnorm1(x)
        x = self.se1(x)
        x = self.depthwiseConv(x)
        x = self.batchnorm2(x)
        x = F.elu(x)
        x = self.se2(x)
        x = self.pool1(x)
        x = self.dropout(x)

        # Block 2
        x = self.separableConv(x)
        x = self.batchnorm3(x)
        x = F.elu(x)
        x = self.se3(x)
        x = self.pool2(x)
        x = self.dropout(x)

        # Flatten and Dense
        x = self.flatten(x)
        x = self.dense(x)
        return F.softmax(x, dim=1)

In [4]:
# Function to compute accuracy
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == targets.squeeze()).sum().item()
            total += targets.size(0)
    
    accuracy = correct / total
    return accuracy*100

# Function to calculate accuracy
def calculate_accuracy(preds, labels):
    _, predicted = torch.max(preds, 1)
    correct = (predicted == labels).sum().item()
    accuracy = correct / labels.size(0)
    return accuracy*100

In [5]:
def eegnet_model_finetuning(train_loader, val_loader, device, verbose=True):
    torch.manual_seed(0)
    # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    learning_rate=1e-3
    num_epochs = 1000
    patience = 30  # Number of epochs to wait before stopping if no improvement
    min_delta = 1e-11 # Minimum change to qualify as improvement

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    model.load_state_dict(torch.load('calibrated_model.pth')) #For Fine Tuning

    # UnFreeze all layers except the last one
    for param in model.parameters():
        param.requires_grad = True  # UnFreeze all parameters
    
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)

    criterion = nn.CrossEntropyLoss().to(device)  # Move loss function to GPU if needed
    # optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    # Training with Early Stopping
    best_val_loss = float('inf')
    early_stop_counter = 0

    # Placeholder for training and validation loss history
    train_losses = []
    val_losses = []
    val_accuracy = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        # Training Loop
        for inputs, targets in train_loader:  # Assuming train_loader is defined
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets.squeeze())

            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()  # Accumulate loss

        # Calculate average training loss for this epoch
        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation loop (set model to evaluation mode)
        model.eval()
        val_loss = 0.0
        val_acc = 0.0

        with torch.no_grad():  # Disable gradient calculation for validation
            for inputs, targets in val_loader:  # Assuming val_loader is defined
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets.squeeze())
                val_loss += loss.item()

                val_acc += calculate_accuracy(outputs, targets.squeeze())

        # Calculate average validation loss for this epoch
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        val_acc /= len(val_loader)
        val_accuracy.append(val_acc)

        if verbose:
            print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}%')

        # Early stopping check
        if val_loss < best_val_loss - min_delta:  # Check if validation loss improved
            best_val_loss = val_loss
            early_stop_counter = 0  # Reset early stop counter
            torch.save(model.state_dict(), 'best_model.pth')  # Save best model
        else:
            early_stop_counter += 1

        
        if early_stop_counter >= patience and epoch > 100:
            if verbose:
                print(f'Early stopping at epoch {epoch+1}')
            break

    # Load the best model before returning
    # model.load_state_dict(torch.load('best_model.pth'))
    # print('Training complete.')

    return model

In [12]:
def eegnet_model_training(train_loader, val_loader, device, from_scratch=True, verbose=True):
    torch.manual_seed(0)
    # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    learning_rate=1e-3
    num_epochs = 1000
    patience = 30  # Number of epochs to wait before stopping if no improvement
    min_delta = 1e-11 # Minimum change to qualify as improvement

    model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
    if not from_scratch:
        # model.load_state_dict(torch.load('best_model.pth'))
        model.load_state_dict(torch.load('calibrated_model.pth')) #For Fine Tuning

        # Freeze all layers except the last one
        for param in model.parameters():
            param.requires_grad = False  # Freeze all parameters

        # Unfreeze the last layer parameters
        # for param in model.dense.parameters():
        #     param.requires_grad = True  # Unfreeze last layer
        
        # Unfreeze the last layer parameters
        for param in model.se1.parameters():
            param.requires_grad = True  # Unfreeze last layer
        
        # Unfreeze the last layer parameters
        for param in model.se2.parameters():
            param.requires_grad = True  # Unfreeze last layer
        
        # Unfreeze the last layer parameters
        for param in model.se3.parameters():
            param.requires_grad = True  # Unfreeze last layer

        
        for param in model.separableConv.parameters():
            param.requires_grad = True

        optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=learning_rate)
    else:
        optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        
    criterion = nn.CrossEntropyLoss().to(device)  # Move loss function to GPU if needed

    # Training with Early Stopping
    best_val_loss = float('inf')
    early_stop_counter = 0

    # Placeholder for training and validation loss history
    train_losses = []
    val_losses = []
    val_accuracy = []

    for epoch in range(num_epochs):
        model.train()
        train_loss = 0.0

        # Training Loop
        for inputs, targets in train_loader:  # Assuming train_loader is defined
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets.squeeze())

            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()  # Accumulate loss

        # Calculate average training loss for this epoch
        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation loop (set model to evaluation mode)
        model.eval()
        val_loss = 0.0
        val_acc = 0.0

        with torch.no_grad():  # Disable gradient calculation for validation
            for inputs, targets in val_loader:  # Assuming val_loader is defined
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets.squeeze())
                val_loss += loss.item()

                val_acc += calculate_accuracy(outputs, targets.squeeze())

        # Calculate average validation loss for this epoch
        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        val_acc /= len(val_loader)
        val_accuracy.append(val_acc)

        if verbose:
            print(f'Epoch {epoch+1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}%')

        # Early stopping check
        if val_loss < best_val_loss - min_delta:  # Check if validation loss improved
            best_val_loss = val_loss
            early_stop_counter = 0  # Reset early stop counter
            torch.save(model.state_dict(), 'best_model.pth')  # Save best model
        else:
            early_stop_counter += 1

        
        if early_stop_counter >= patience and epoch > 100:
            if verbose:
                print(f'Early stopping at epoch {epoch+1}')
            break

    # Load the best model before returning
    # model.load_state_dict(torch.load('best_model.pth'))
    # print('Training complete.')

    return model

In [7]:
def eegnet_model_evaluation(model, device, test_loader):
    # Validation loop (set model to evaluation mode)
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():  # Disable gradient calculation for validation
        for inputs, targets in test_loader:  # Assuming val_loader is defined
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)

            # Get predictions and calculate accuracy
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets.squeeze()).sum().item()

            # test_acc = calculate_accuracy(outputs, targets.squeeze())
    
        # Calculate average loss and accuracy
    test_acc =  100*correct / total

    return test_acc

In [8]:
torch.manual_seed(0)
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

batch_size=32
fs = 500
train_ratio = 0.9
sub = 7 #Till subject 7 (starting from 0), only calibration sessions are conducted

# Xtr, Ytr = create_dataset(sub, base_path=parent_dir)
Xtr, Ytr = utils.calib_sess_dataset(base_path=parent_dir)
X_train = utils.baseline_correction(Xtr)
X_train = utils.bandpass_filtering(X_train)

# Creating train-validation split
eeg_train, eeg_val, label_train, label_val = train_test_split(X_train, Ytr, 
                                                              train_size=train_ratio, random_state=42, shuffle=True)


X_train_tensor = torch.tensor(eeg_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
Y_train_tensor = torch.tensor(label_train, dtype=torch.long).to(device)  # Use long for classification

X_val_tensor = torch.tensor(eeg_val, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
Y_val_tensor = torch.tensor(label_val, dtype=torch.long).to(device)

train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
test_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

model = eegnet_model_training(train_loader, val_loader, device)
torch.save(model.state_dict(), 'calibrated_model.pth')  # Save best model


d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\init.py:453: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")
d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:134: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\modules\conv.py:454: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\Convolution.cpp:1032.)
  return F.conv2d(input, weight, bias, self.stride,


Epoch 1/1000, Train Loss: 0.6954, Val Loss: 0.6930, Val Acc: 56.0033%
Epoch 2/1000, Train Loss: 0.6776, Val Loss: 0.6939, Val Acc: 50.7401%
Epoch 3/1000, Train Loss: 0.6637, Val Loss: 0.7021, Val Acc: 42.3520%
Epoch 4/1000, Train Loss: 0.6628, Val Loss: 0.7007, Val Acc: 43.9145%
Epoch 5/1000, Train Loss: 0.6484, Val Loss: 0.7033, Val Acc: 46.5461%
Epoch 6/1000, Train Loss: 0.6393, Val Loss: 0.7019, Val Acc: 53.3717%
Epoch 7/1000, Train Loss: 0.6362, Val Loss: 0.6980, Val Acc: 53.3717%
Epoch 8/1000, Train Loss: 0.6284, Val Loss: 0.6920, Val Acc: 50.7401%
Epoch 9/1000, Train Loss: 0.6123, Val Loss: 0.6857, Val Acc: 52.8783%
Epoch 10/1000, Train Loss: 0.6043, Val Loss: 0.6731, Val Acc: 58.1414%
Epoch 11/1000, Train Loss: 0.5696, Val Loss: 0.6411, Val Acc: 68.5855%
Epoch 12/1000, Train Loss: 0.5579, Val Loss: 0.6363, Val Acc: 62.8289%
Epoch 13/1000, Train Loss: 0.5599, Val Loss: 0.6250, Val Acc: 65.9539%
Epoch 14/1000, Train Loss: 0.5506, Val Loss: 0.6175, Val Acc: 62.2533%
Epoch 15/1000, 

In [9]:
torch.manual_seed(0)
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size=32

model = EEGNet(nb_classes=2, Chans=27, Samples=2000).to(device)
model.load_state_dict(torch.load('calibrated_model.pth'))
for sub in range(8, 21):
    Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)

    X_test = utils.baseline_correction(Xte)
    X_test = utils.bandpass_filtering(X_test)

    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_test_tensor = torch.tensor(Yte, dtype=torch.long).to(device)
    test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

    test_acc = eegnet_model_evaluation(model, device, test_loader)
    
    print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')

d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\init.py:453: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")
d:\Praveen\PostDoc@SIT\.venv\Lib\site-packages\torch\nn\utils\weight_norm.py:134: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
C:\Users\postd\AppData\Local\Temp\ipykernel_10380\3907910491.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitr

Subject: 08, Test Accuracy: 47.92%
Subject: 09, Test Accuracy: 56.25%
Subject: 10, Test Accuracy: 50.00%
Subject: 11, Test Accuracy: 43.75%
Subject: 12, Test Accuracy: 64.58%
Subject: 13, Test Accuracy: 52.08%
Subject: 14, Test Accuracy: 66.67%
Subject: 15, Test Accuracy: 52.08%
Subject: 16, Test Accuracy: 54.17%
Subject: 17, Test Accuracy: 58.33%
Subject: 18, Test Accuracy: 50.00%
Subject: 19, Test Accuracy: 54.17%
Subject: 20, Test Accuracy: 62.50%


In [13]:
# Model Trained From Scratch using Subject specific calibration session data and subject that participated in calibration only. 
parent_dir = os.path.dirname(os.getcwd())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

Xcalib, Ycalib = utils.calib_sess_dataset(base_path=parent_dir)

train_ratio = 0.9
batch_size=32
perf = dict()
for sub in range(8, 21):
    Xtr, Ytr, Xte, Yte = utils.online_sess_dataset(sub, base_path=parent_dir)
    eeg_train, eeg_val, label_train, label_val = train_test_split(Xtr, Ytr, 
                                                              train_size=train_ratio, random_state=42, shuffle=True)
    
    # Xtrain = np.concatenate((Xcalib, eeg_train), axis=0)
    # Ytrain = np.concatenate((Ycalib, label_train), axis=0)

    Xtrain = eeg_train
    Ytrain = label_train

    X_train = utils.baseline_correction(Xtrain)
    X_train = utils.bandpass_filtering(X_train)

    Xval = utils.baseline_correction(eeg_val)
    Xval = utils.bandpass_filtering(Xval)
    

    X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_train_tensor = torch.tensor(Ytrain, dtype=torch.long).to(device)  # Use long for classification

    X_val_tensor = torch.tensor(Xval, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_val_tensor = torch.tensor(label_val, dtype=torch.long).to(device)

    train_dataset = TensorDataset(X_train_tensor, Y_train_tensor)
    test_dataset = TensorDataset(X_val_tensor, Y_val_tensor)

    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=True)

    # model = eegnet_model_finetuning(train_loader, val_loader, device, verbose=False)
    # model = eegnet_model_training(train_loader, val_loader, device, from_scratch=True, verbose=False) # Training From Scratch
    model = eegnet_model_training(train_loader, val_loader, device, from_scratch=False, verbose=False) # Fine Tuning


    # --------------------------------------------------------------------------- #
    X_test = utils.baseline_correction(Xte)
    X_test = utils.bandpass_filtering(X_test)

    X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(1).permute(0, 1, 3, 2).to(device)  # Shape: (batch_size, 1, 27, 2500)
    Y_test_tensor = torch.tensor(Yte, dtype=torch.long).to(device)
    test_dataset = TensorDataset(X_test_tensor, Y_test_tensor)

    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

    test_acc = eegnet_model_evaluation(model, device, test_loader)
    perf[f'Sub{sub}'] = test_acc
    
    print(f'Subject: {sub:02d}, Test Accuracy: {test_acc:.2f}%')

    # Xcalib = np.concatenate((Xcalib, Xtr, Xte), axis=0)
    # Ycalib = np.concatenate((Ycalib, Ytr, Yte), axis=0)

C:\Users\postd\AppData\Local\Temp\ipykernel_10380\2779258201.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('calibrated_model.pth')) #

Subject: 08, Test Accuracy: 52.08%
Subject: 09, Test Accuracy: 56.25%
Subject: 10, Test Accuracy: 60.42%
Subject: 11, Test Accuracy: 50.00%
Subject: 12, Test Accuracy: 54.17%
Subject: 13, Test Accuracy: 60.42%
Subject: 14, Test Accuracy: 43.75%
Subject: 15, Test Accuracy: 45.83%
Subject: 16, Test Accuracy: 54.17%
Subject: 17, Test Accuracy: 52.08%
Subject: 18, Test Accuracy: 47.92%
Subject: 19, Test Accuracy: 68.75%
Subject: 20, Test Accuracy: 70.83%


In [14]:
mean_acc = list(perf.values())
print(mean_acc)
print(f'Average Accuracy: {np.mean(mean_acc)}')

[52.083333333333336, 56.25, 60.416666666666664, 50.0, 54.166666666666664, 60.416666666666664, 43.75, 45.833333333333336, 54.166666666666664, 52.083333333333336, 47.916666666666664, 68.75, 70.83333333333333]
Average Accuracy: 55.12820512820513


In [11]:
mean_acc = list(perf.values())
print(mean_acc)
print(f'Average Accuracy: {np.mean(mean_acc)}')

[47.916666666666664, 52.083333333333336, 52.083333333333336, 54.166666666666664, 54.166666666666664, 60.416666666666664, 50.0, 50.0, 58.333333333333336, 45.833333333333336, 50.0, 62.5, 83.33333333333333]
Average Accuracy: 55.44871794871795


Reduction Rate = 32,

Transfer Learning Layer - SE3, Dense Layer
[54.166666666666664, 56.25, 52.083333333333336, 54.166666666666664, 56.25, 58.333333333333336, 60.416666666666664, 43.75, 50.0, 47.916666666666664, 56.25, 54.166666666666664, 68.75]
Average Accuracy: 54.8076923076923
---------------------------------------

Transfer Learning Layer - SE3, Dense, Last Conv
[60.416666666666664, 54.166666666666664, 45.833333333333336, 54.166666666666664, 60.416666666666664, 56.25, 50.0, 47.916666666666664, 56.25, 41.666666666666664, 62.5, 50.0, 77.08333333333333]
Average Accuracy: 55.128205128205124

In [ ]:
1. Retraining the EEGNet from Scratch
2. Fine Tuning the EEGNet on new subject's data
3. Variants of fine tuning - last layer alone, SE Nets etc.

4. Retraining the EEGNet with SE from scratch
5. Fine Tuning EEGNet-SE on new subjects' data
6. Variants of fine tuning - last layer alone,SE Nets etc.